# Feature-Engineered Data Quality Check

Load the feature-engineered parquet dataset from S3 with `LoaderStorage`, convert it into a Polars dataframe, and run generic data quality checks for missing values, non-finite values, duplicates, constant columns, blank strings, and numeric ranges.

In [ ]:
from __future__ import annotations

import os

import polars as pl
from IPython.display import display

from loader import LoaderStorage

## Dataset Location

The S3 backend requires `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`. If the bucket uses a custom S3-compatible endpoint, set `S3_ENDPOINT_URL` or `MLFLOW_S3_ENDPOINT_URL` before running the notebook.

In [ ]:
# Change these paths to your actual dataset location.
DATA_ROOT = "s3://data-mining"
INPUT_PATH = "data/features/feature_engineered.parquet"

missing_env = [
    name
    for name in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
    if DATA_ROOT.startswith("s3://") and not os.environ.get(name)
]
if missing_env:
    raise RuntimeError(f"Missing S3 environment variables: {', '.join(missing_env)}")

storage = LoaderStorage(DATA_ROOT)
dataset_path = storage.path(INPUT_PATH)
dataset_path

## Load Parquet As Polars

`LoaderStorage.read_parquet()` returns a pandas dataframe. To keep the actual working dataframe in Polars, this uses the loader's resolved path and S3 filesystem handle directly with `pl.read_parquet()`.

In [ ]:
if storage.fs is not None:
    with storage.fs.open(dataset_path, "rb") as parquet_file:
        df = pl.read_parquet(parquet_file)
else:
    df = pl.read_parquet(dataset_path)

print(f"Loaded {df.height:,} rows x {df.width:,} columns")
print(f"Estimated size: {df.estimated_size('mb'):,.2f} MB")
display(df.head())

In [ ]:
schema_report = pl.DataFrame(
    {
        "column": list(df.schema.keys()),
        "dtype": [str(dtype) for dtype in df.schema.values()],
    }
)
display(schema_report)
display(df.describe())

## Missing, NaN, And Infinite Values

In [ ]:
row_count = max(df.height, 1)
float_dtypes = {pl.Float32, pl.Float64}
float_columns = [column for column, dtype in df.schema.items() if dtype in float_dtypes]

null_counts = (
    df.null_count()
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
    .with_columns((pl.col("null_count") / row_count).alias("null_rate"))
    .sort("null_count", descending=True)
)
null_counts_nonzero = null_counts.filter(pl.col("null_count") > 0)

if float_columns:
    nan_counts = (
        df.select([pl.col(column).is_nan().sum().alias(column) for column in float_columns])
        .transpose(include_header=True, header_name="column", column_names=["nan_count"])
        .with_columns((pl.col("nan_count") / row_count).alias("nan_rate"))
        .sort("nan_count", descending=True)
    )
    infinite_counts = (
        df.select([pl.col(column).is_infinite().sum().alias(column) for column in float_columns])
        .transpose(include_header=True, header_name="column", column_names=["infinite_count"])
        .with_columns((pl.col("infinite_count") / row_count).alias("infinite_rate"))
        .sort("infinite_count", descending=True)
    )
else:
    nan_counts = pl.DataFrame({"column": [], "nan_count": [], "nan_rate": []})
    infinite_counts = pl.DataFrame({"column": [], "infinite_count": [], "infinite_rate": []})

display(null_counts_nonzero)
display(nan_counts.filter(pl.col("nan_count") > 0))
display(infinite_counts.filter(pl.col("infinite_count") > 0))

## Duplicates, Constants, And Blank Strings

In [ ]:
duplicate_columns = sorted({column for column in df.columns if df.columns.count(column) > 1})
duplicate_row_count = int(df.is_duplicated().sum()) if df.height else 0

unique_counts = (
    df.select([pl.col(column).n_unique().alias(column) for column in df.columns])
    .transpose(include_header=True, header_name="column", column_names=["unique_count"])
    .sort("unique_count")
)
constant_columns = unique_counts.filter(pl.col("unique_count") <= 1)

string_columns = [column for column, dtype in df.schema.items() if dtype == pl.String]
if string_columns:
    blank_string_counts = (
        df.select([
            pl.col(column).str.strip_chars().eq("").sum().alias(column)
            for column in string_columns
        ])
        .transpose(include_header=True, header_name="column", column_names=["blank_string_count"])
        .with_columns((pl.col("blank_string_count") / row_count).alias("blank_string_rate"))
        .sort("blank_string_count", descending=True)
    )
else:
    blank_string_counts = pl.DataFrame({"column": [], "blank_string_count": [], "blank_string_rate": []})

print(f"Duplicate columns: {duplicate_columns or 'none'}")
print(f"Duplicate rows: {duplicate_row_count:,}")
display(constant_columns)
display(blank_string_counts.filter(pl.col("blank_string_count") > 0))

## Numeric Range Checks

Review the minimum, maximum, mean, standard deviation, zero counts, and negative counts. Negative values are not automatically wrong because some engineered features can legitimately be negative; treat this table as a quick sanity scan.

In [ ]:
numeric_dtypes = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}
numeric_columns = [column for column, dtype in df.schema.items() if dtype in numeric_dtypes]

numeric_profile_rows = []
for column in numeric_columns:
    series = df.get_column(column)
    numeric_profile_rows.append(
        {
            "column": column,
            "dtype": str(series.dtype),
            "min": series.min(),
            "max": series.max(),
            "mean": series.mean(),
            "std": series.std(),
            "zero_count": int((series == 0).sum()) if df.height else 0,
            "negative_count": int((series < 0).sum()) if df.height else 0,
        }
    )

numeric_profile = pl.DataFrame(numeric_profile_rows) if numeric_profile_rows else pl.DataFrame()
display(numeric_profile)

## Quality Flag Summary

In [ ]:
issues = []

if df.height == 0:
    issues.append({"severity": "error", "check": "row_count", "details": "Dataframe is empty"})
if duplicate_columns:
    issues.append({"severity": "error", "check": "duplicate_columns", "details": str(duplicate_columns)})
if duplicate_row_count:
    issues.append({"severity": "warning", "check": "duplicate_rows", "details": f"{duplicate_row_count:,} duplicate rows"})

for row in null_counts_nonzero.iter_rows(named=True):
    issues.append({
        "severity": "warning",
        "check": "null_values",
        "details": f"{row['column']}: {row['null_count']:,} nulls ({row['null_rate']:.2%})",
    })
for row in nan_counts.filter(pl.col("nan_count") > 0).iter_rows(named=True):
    issues.append({
        "severity": "error",
        "check": "nan_values",
        "details": f"{row['column']}: {row['nan_count']:,} NaNs ({row['nan_rate']:.2%})",
    })
for row in infinite_counts.filter(pl.col("infinite_count") > 0).iter_rows(named=True):
    issues.append({
        "severity": "error",
        "check": "infinite_values",
        "details": f"{row['column']}: {row['infinite_count']:,} infinite values ({row['infinite_rate']:.2%})",
    })
for row in constant_columns.iter_rows(named=True):
    issues.append({
        "severity": "warning",
        "check": "constant_column",
        "details": f"{row['column']}: {row['unique_count']} unique value(s)",
    })
for row in blank_string_counts.filter(pl.col("blank_string_count") > 0).iter_rows(named=True):
    issues.append({
        "severity": "warning",
        "check": "blank_strings",
        "details": f"{row['column']}: {row['blank_string_count']:,} blank strings ({row['blank_string_rate']:.2%})",
    })

if issues:
    issue_report = pl.DataFrame(issues)
else:
    issue_report = pl.DataFrame({
        "severity": ["ok"],
        "check": ["generic_quality_checks"],
        "details": ["No obvious issues found by these generic checks"],
    })

display(issue_report)